# Importint the libraries

In [ ]:
import MetaTrader5 as mt
from datetime import datetime
import pandas as pd
import numpy as np
import pandas_ta as ta
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.feature_selection import RFE
from sklearn.model_selection import TimeSeriesSplit
from sklearn.impute import SimpleImputer
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning) 
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Login

In [ ]:
mt.intialize()
username = int(os.environ['MT5_LOGIN'])
password = os.environ['MT5_PASSWORD']
server = 'Alpari-MT5-Demo'
mt.login(username, password, server)

# Get data

In [ ]:
ticker = 'EURUSD'
interval = mt.TIMEFRAME_H4
from_date = datetime.now()
no_of_rows = 100
df = mt.copy_rates_from(ticker, interval, from_date, no_of_rows)
df
# ohlc = pd.DataFrame(mt.copy_rates_range('EURUSD', mt.TIMEFRAME_M1, datetime(2023,7,21), datetime.now()))
# ohlc

# Data process

In [ ]:
df[["lowerBB2", "midBB","upperBB2","bandwidthBB2","percentBB2"]] = ta.bbands(df["Close"], length=21, std=2)
df["returns"] = np.log(df.Close.div(df.Close.shift(1)))
df.dropna(inplace=True)

In [ ]:
df["min_6"] = df['Close'].rolling(6).min() / df['Close'] - 1
df["max_6"] = df['Close'].rolling(6).max() / df['Close'] - 1
df["boll_6"] = (df['Close'] - df['Close'].rolling(6).mean()) / df['Close'].rolling(6).std()
df["min"] = df['Close'].rolling(30).min() / df['Close'] - 1
df["max"] = df['Close'].rolling(30).max() / df['Close'] - 1
df["boll"] = (df['Close'] - df['Close'].rolling(30).mean()) / df['Close'].rolling(30).std()
df['min_6 / min'] = df['min_6'] / df['min']
df['max_6 / max'] = df['max_6'] / df['max']
df['min / max'] = df['min'] / df['max']
df['min_6 / max_6'] = df['min_6'] / df['max_6']
df['boll_6 / boll'] = df['boll_6'] / df['boll']
    
# Replace infinities and handle NaNs
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Ensure columns exist and impute missing values
columns_to_impute = ['min_6 / max_6', 'min / max', 'boll_6 / boll']
imputer = SimpleImputer(strategy='mean')
for col in columns_to_impute:
    if col in df.columns:
        df[[col]] = imputer.fit_transform(df[[col]])

# Final cleanup (if needed)
df.dropna(inplace=True)


In [ ]:
df["target"] = np.where(df['returns'] > 0, 1,np.where(df['returns'] < 0, -1,np.nan))
df.dropna(inplace=True)

In [ ]:
lags = 1
features = []
attributes = df.drop(columns=['Open','High','Low','Close',"lowerBB2", "midBB","upperBB2","bandwidthBB2",'target'])
for a in attributes:
        for lag in range(1, lags + 1):
            col = "{}_lag_{}".format(a, lag)
            features.append(col)
            df[col] = df[a].shift(lag)
df.dropna(inplace = True)

# Get signal

In [ ]:
X = df[features]
y = df["target"]
model = joblib.load('U.joblib')
y_pred = model.predict(X)
df['pred'] = model.predict(X)

# Execute the position

In [ ]:
ticker = 'EURUSD'
qty = 0.01
buy_order_type = mt. ORDER_TYPE_BUY
sell_order_type = me.okoen_tresseLe
buy_price = mt.symbol_info_tick("BTCUSD").ask
sell_price = mt.symbol_info_tick("BTCUSD").bid
s1_pct = 0.05
tp_pct = 0.1
buy_sl = buy_price * (1-s1_pct)
buy_tp = buy_price * (1+tp_pct)
sell_sl = sell_price * (1+s1_pct)
sell_tp = sell_price * (1-tp_pct)

def create_order(ticker, qty, order_type, price, sl, tp):
    request = {
    "action": mt.TRADE_ACTION_DEAL,
    "symbol": ticker,
    "volume": qty,
    "type": order_type,
    "price":price,
    "sl": sl,
    "tp": tp,
    "comment": "python open",
    "type_time": mt.ORDER_TIME_GTC,
    "type_filling": mt.ORDER_FILLING_IOC,
}
    order = mt.order_send(request)
    return order

def close_order(ticker, qty, order_type, price):
    request = {
    "action": mt.TRADE_ACTION_DEAL,
    "symbol": ticker,
    "volume": qty,
    "type": order_type,
    "position": mt.positions_get()[0]._asdict()['ticket'],
    "price": price,
    "comment": "close the position",
    "type_time": mt.ORDER_TIME_GTC,
    "type_filling": mt.ORDER_FILLING_IOC,
}
    order = mt.order_send(request)
    return order